# Verificación contra un resultado publicado — Card y Krueger (1994)

**Unidad 4.d** · Acompaña a `Clase_11_DiferenciaEnDiferencias` · Notas: cap. 9

El propósito no es estimar el efecto del salario mínimo —eso ya lo hace el cuaderno de la
Clase 11— sino **mostrar cómo se verifica** un resultado, contra un número que alguien
más publicó y que podemos consultar.

> **La regla de trabajo del curso:** un resultado que no se puede contrastar contra algo
> —un artículo, otra implementación, una identidad matemática— no se reporta.

Card y Krueger (1994), cuadro 3: el estimador de diferencia en diferencias del empleo
equivalente de tiempo completo es **+2.76** empleados por establecimiento (Nueva Jersey
frente a Pensilvania, febrero a noviembre de 1992).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PUBLICADO = 2.76  # Card y Krueger (1994), cuadro 3

datos = pd.read_csv("../../Clase_11_DiferenciaEnDiferencias/employment.csv")

print(f"Establecimientos: {len(datos)}")
print(datos.groupby("state").size().rename("tiendas").to_string())
print("  (0 = Pensilvania, control · 1 = Nueva Jersey, tratado)")

## Vía 1 — diferencia de diferencias de medias

Es la definición del capítulo 9: la diferencia entre el cambio del grupo tratado y el
cambio del grupo de control.

In [ ]:
medias = datos.groupby("state")[["total_emp_feb", "total_emp_nov"]].mean()
cambio = medias["total_emp_nov"] - medias["total_emp_feb"]
did = cambio.loc[1] - cambio.loc[0]

print("Empleo promedio por establecimiento:")
print(medias.round(4).to_string())
print("\nCambio febrero -> noviembre:")
print(f"  Pensilvania (control) : {cambio.loc[0]:+.4f}")
print(f"  Nueva Jersey (tratado): {cambio.loc[1]:+.4f}")
print(f"\n  DiD = {cambio.loc[1]:+.4f} - ({cambio.loc[0]:+.4f}) = {did:+.4f}")

## Vía 2 — el coeficiente de interacción en una regresión

El mismo estimador, calculado de otra manera. Deben coincidir: es la equivalencia
algebraica del capítulo 9.

In [ ]:
largo = datos.reset_index(names="tienda").melt(
    id_vars=["tienda", "state"],
    value_vars=["total_emp_feb", "total_emp_nov"],
    var_name="periodo",
    value_name="empleo",
)
largo["post"] = (largo["periodo"] == "total_emp_nov").astype(int)
largo["tratado"] = largo["state"]

modelo = smf.ols("empleo ~ tratado + post + tratado:post", data=largo).fit()
agrupado = modelo.get_robustcov_results(cov_type="cluster", groups=largo["tienda"])
coef = modelo.params["tratado:post"]

print(f"  beta_(tratado x post)  = {coef:+.4f}")
print(f"  ee sin agrupar         = {modelo.bse['tratado:post']:.4f}")
print(f"  ee agrupado por tienda = {agrupado.bse[3]:.4f}")

print(f"\n  |{did:.6f} - {coef:.6f}| = {abs(did - coef):.2e}", end="  ")
print("IDÉNTICAS" if np.isclose(did, coef) else "DIFIEREN — hay un error")

## Verificación contra el artículo

In [ ]:
diferencia = abs(did - PUBLICADO)

print(f"  estimado por nosotros : {did:+.4f}")
print(f"  publicado             : {PUBLICADO:+.4f}")
print(f"  diferencia            : {diferencia:.4f}")

if diferencia < 0.05:
    print("\n  REPLICA. La diferencia (0.01) es de redondeo: el artículo reporta")
    print("  dos decimales y depura la muestra de manera ligeramente distinta.")
else:
    print("\n  NO REPLICA. Revisar codificación de estados, definición del empleo")
    print("  equivalente, y qué establecimientos se excluyen.")

## Un detalle que suele enseñarse al revés

Se repite mucho que «agrupar los errores estándar los agranda». Veamos qué pasa aquí:

In [ ]:
razon_inter = agrupado.bse[3] / modelo.bse["tratado:post"]
razon_trat = agrupado.bse[1] / modelo.bse["tratado"]

print(f"  interacción : {modelo.bse['tratado:post']:.4f} -> {agrupado.bse[3]:.4f}"
      f"   (x{razon_inter:.2f})")
print(f"  tratado     : {modelo.bse['tratado']:.4f} -> {agrupado.bse[1]:.4f}"
      f"   (x{razon_trat:.2f})")

Agrupar **reduce** el error estándar de la interacción y **aumenta** el de `tratado`. No
es un error.

La afirmación habitual es cierta sólo para regresores que varían **entre**
conglomerados: `tratado` es fijo dentro de cada tienda, y la correlación positiva de sus
observaciones infla la varianza del estimador.

La interacción es de otra naturaleza: es un **contraste temporal dentro** de la tienda.
La correlación positiva del empleo de una misma tienda entre febrero y noviembre se
cancela al diferenciar, y eso reduce la varianza del contraste.

**La dirección del cambio depende de si el regresor varía dentro o entre conglomerados;
hay que verificarla, no suponerla.**

## Los tres controles de esta plantilla

Para cualquier resultado propio, buscar los tres que aquí se aplicaron:

1. **Dos implementaciones independientes** que deban coincidir (medias y regresión).
2. **Un número externo publicado** con el cual contrastar.
3. **Errores estándar al nivel correcto** de agrupamiento.

Cuando no exista un número publicado —el caso habitual en un trabajo original— quedan el
1 y el 3, más las identidades matemáticas del cuadro de
[`lista_de_verificacion.md`](../05_Reproducibilidad/lista_de_verificacion.md).

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las demás actividades.